# 03 - 模型推理与评估

加载训练好的模型，对比物理仿真 GT 与模型预测的 3D 形态。

In [ ]:
import sys, os, glob
import numpy as np
import torch
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from ipywidgets import interact, IntSlider
from tqdm import tqdm

%matplotlib inline

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# GPU 设置
CUDA_DEVICE = 0
os.environ['CUDA_VISIBLE_DEVICES'] = str(CUDA_DEVICE)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. 配置参数

In [ ]:
# --- 修改以下路径 ---

# 模型权重所在目录（包含 model/best_model.pt 和 action_norm_factor.txt）
LOG_DIR = os.path.join(PROJECT_ROOT, 'train_log', 'train_log_seq_vis', 'experiment_2')

# 测试数据（与训练不同的序列文件）
DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'sequence_data')

# 模型选择: 'v1', 'v2', 'v4_nerf', 'v5_deformable'
MODEL_VERSION = 'v2'

# 推理设置
SEQ_LEN = 20          # 必须与训练时一致
DENSITY_THRESHOLD = 0.75
DEMO_LENGTH = 50       # 推理帧数
SIM_STEPS_PER_ACTION = 500

print(f'模型版本: {MODEL_VERSION}')
print(f'权重目录: {LOG_DIR}')

## 2. 加载模型

In [ ]:
# 加载归一化系数
norm_path = os.path.join(LOG_DIR, 'action_norm_factor.txt')
if os.path.exists(norm_path):
    norm_factor = float(np.loadtxt(norm_path))
else:
    norm_factor = 1.0
    print('Warning: 使用默认 norm_factor=1.0')
print(f'Norm factor: {norm_factor}')

# 加载测试数据
seq_files = sorted(glob.glob(os.path.join(DATA_DIR, '*.npz')))
if not seq_files:
    raise FileNotFoundError(f'无数据: {DATA_DIR}')
test_file = seq_files[-1]
test_data = np.load(test_file)
actions_raw = test_data['actions']
action_dim = actions_raw.shape[1]
print(f'测试数据: {os.path.basename(test_file)}, 动作维度: {action_dim}')

# 初始化模型
if MODEL_VERSION == 'v1':
    from src.models.model_seq import model_v1 as ModelClass
    model = ModelClass(action_dim=action_dim, seq_len=SEQ_LEN, hidden_dim=256)
elif MODEL_VERSION == 'v2':
    from src.models.model_seq_skip import model_v2 as ModelClass
    model = ModelClass(action_dim=action_dim, seq_len=SEQ_LEN, hidden_dim=256)
elif MODEL_VERSION == 'v4_nerf':
    from src.models.model_v4_nerf_pinn import NeRF_PINN as ModelClass
    model = ModelClass(action_dim=action_dim, seq_len=SEQ_LEN, hidden_dim=128)
elif MODEL_VERSION == 'v5_deformable':
    from src.models.model_v5_deformable import DeformableSoftRobotModel as ModelClass
    model = ModelClass(action_dim=action_dim, seq_len=SEQ_LEN, hidden_dim=128)
else:
    raise ValueError(f'Unknown model version: {MODEL_VERSION}')

# 加载权重
weight_path = os.path.join(LOG_DIR, 'model', 'best_model.pt')
model.load_state_dict(torch.load(weight_path, map_location=device))
model.to(device).eval()
print(f'模型加载完成: {weight_path}')

## 3. 准备 3D 查询网格

In [ ]:
GRID_RES = 30
x = np.linspace(-0.3, 0.3, GRID_RES)
y = np.linspace(-0.3, 0.3, GRID_RES)
z = np.linspace(0.0, 0.6, GRID_RES)

gx, gy, gz = np.meshgrid(x, y, z, indexing='ij')
grid_points = np.stack([gx.flatten(), gy.flatten(), gz.flatten()], axis=-1)
grid_tensor = torch.tensor(grid_points, dtype=torch.float32, device=device)
print(f'查询网格: {grid_points.shape[0]} 个点 ({GRID_RES}^3)')

## 4. 运行推理

In [ ]:
from elastica_env import ContinuousSoftArmEnv

env = ContinuousSoftArmEnv(dt=1e-4)
history_buffer = torch.zeros((1, SEQ_LEN, action_dim), device=device)

gt_positions = []
pred_clouds = []
demo_length = min(DEMO_LENGTH, len(actions_raw))

for i in tqdm(range(demo_length), desc='推理中'):
    target_action = actions_raw[i]

    # --- GT: 物理仿真 ---
    env.set_action(target_action)
    for _ in range(SIM_STEPS_PER_ACTION):
        env.step()
    rod = env.simulation[0]
    gt_positions.append(rod.position_collection.copy().T)

    # --- Pred: 模型推理 ---
    act_norm = target_action / norm_factor
    act_tensor = torch.tensor(act_norm, dtype=torch.float32, device=device).view(1, 1, -1)
    history_buffer = torch.cat([history_buffer[:, 1:, :], act_tensor], dim=1)

    with torch.no_grad():
        if MODEL_VERSION in ('v1', 'v2'):
            state = model.encode_temporal(history_buffer)
            pts_input = grid_tensor.unsqueeze(0)
            if MODEL_VERSION == 'v2':
                curr_act = history_buffer[:, -1, :]
                raw_out = model.decode_spatial(pts_input, state, curr_act)
            else:
                raw_out = model.decode_spatial(pts_input, state)
            density = 1.0 - torch.exp(-torch.nn.functional.relu(raw_out[0, :, 0]))
        elif MODEL_VERSION == 'v5_deformable':
            latent_seq = model.get_physics_state(history_buffer)
            current_state = latent_seq[:, -1, :]
            state_rep = current_state.repeat_interleave(grid_tensor.shape[0], dim=0)
            density, _ = model.query_field(grid_tensor.unsqueeze(0), current_state)
            density = density.squeeze(-1)[0]
        else:
            # v4_nerf
            latent_seq = model.get_physics_state(history_buffer)
            current_state = latent_seq[:, -1, :]
            current_action = history_buffer[:, -1, :]
            raw_out = model.forward_rendering(
                grid_tensor.unsqueeze(0),
                current_state.repeat_interleave(grid_tensor.shape[0], dim=0),
                current_action.repeat_interleave(grid_tensor.shape[0], dim=0),
            )
            density = 1.0 - torch.exp(-torch.nn.functional.relu(raw_out[0, :, 1]))

    mask = density > DENSITY_THRESHOLD
    pred_clouds.append(grid_points[mask.cpu().numpy()])

print(f'\n推理完成: {demo_length} 帧')

## 5. 交互式对比浏览

In [ ]:
from IPython.display import display, clear_output
import ipywidgets as widgets

slider = widgets.IntSlider(min=0, max=demo_length-1, step=1, value=0, description='Frame')
out = widgets.Output()

def on_slider_change(change):
    with out:
        clear_output(wait=True)
        frame_idx = slider.value
        fig = plt.figure(figsize=(18, 5))
        ax1 = fig.add_subplot(131, projection='3d')
        pos = gt_positions[frame_idx]
        ax1.plot(pos[:, 0], pos[:, 1], pos[:, 2], 'b-', linewidth=4)
        ax1.set_xlim(-0.3, 0.3); ax1.set_ylim(-0.3, 0.3); ax1.set_zlim(0, 0.6)
        ax1.set_title(f'GT (Frame {frame_idx})')
        ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')
        ax2 = fig.add_subplot(132)
        clrs = ['tab:red', 'tab:blue', 'tab:green']
        for d in range(actions_raw.shape[1]):
            ax2.plot(actions_raw[:demo_length, d], color=clrs[d % 3], alpha=0.5, label=f'Action {d}')
        ax2.axvline(x=frame_idx, color='k', linestyle='--', linewidth=2)
        ax2.legend(); ax2.set_title('Driving Actions'); ax2.grid(True, alpha=0.3)
        ax3 = fig.add_subplot(133, projection='3d')
        cloud = pred_clouds[frame_idx]
        if len(cloud) > 0:
            ax3.scatter(cloud[:, 0], cloud[:, 1], cloud[:, 2], c='r', s=3, alpha=0.5)
        ax3.set_xlim(-0.3, 0.3); ax3.set_ylim(-0.3, 0.3); ax3.set_zlim(0, 0.6)
        ax3.set_title(f'Pred ({len(cloud)} pts)')
        ax3.set_xlabel('X'); ax3.set_ylabel('Y'); ax3.set_zlabel('Z')
        plt.tight_layout()
        plt.show()

slider.observe(on_slider_change, names='value')
display(slider, out)
on_slider_change(None)

## 6. 网格概览

In [ ]:
N_COLS = 5
N_ROWS = 2
step = max(1, demo_length // (N_COLS * N_ROWS))
frame_indices = list(range(0, demo_length, step))[:N_COLS * N_ROWS]

fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(4*N_COLS, 4*N_ROWS),
                         subplot_kw={'projection': '3d'})

for ax, fi in zip(axes.flat, frame_indices):
    # GT (蓝色线)
    pos = gt_positions[fi]
    ax.plot(pos[:, 0], pos[:, 1], pos[:, 2], 'b-', linewidth=3, label='GT')
    # Pred (红色点)
    cloud = pred_clouds[fi]
    if len(cloud) > 0:
        ax.scatter(cloud[:, 0], cloud[:, 1], cloud[:, 2], c='r', s=2, alpha=0.4)
    ax.set_xlim(-0.3, 0.3); ax.set_ylim(-0.3, 0.3); ax.set_zlim(0, 0.6)
    ax.set_title(f'Frame {fi}', fontsize=9)
    ax.set_xticklabels([]); ax.set_yticklabels([]); ax.set_zticklabels([])

plt.suptitle(f'GT (blue) vs Pred (red) — {MODEL_VERSION}', fontsize=14)
plt.tight_layout()
plt.show()

## 7. 统计分析

In [ ]:
n_points_per_frame = [len(c) for c in pred_clouds]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(n_points_per_frame)
ax1.set_title('Predicted Points per Frame')
ax1.set_xlabel('Frame')
ax1.set_ylabel('Number of Points')
ax1.grid(True, alpha=0.3)

# GT 端点位置追踪
ee_positions = np.array([pos[-1] for pos in gt_positions])  # 末端节点
ax2.plot(ee_positions[:, 0], label='X', color='r')
ax2.plot(ee_positions[:, 1], label='Y', color='g')
ax2.plot(ee_positions[:, 2], label='Z', color='b')
ax2.set_title('GT End-Effector Position')
ax2.set_xlabel('Frame')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()